# Data Cleaning dan Feature Engineering

Notebook ini menjalankan proses pembersihan data dan pembuatan fitur dari hasil scraping mentah Glints di `data/raw/job_details.csv` menjadi dataset siap pakai di `data/processed/master_dataset_eda.csv`.

---

## Ringkasan Alur Data

- File input berasal dari `data/raw/job_details.csv` yang berisi deskripsi pekerjaan, teks gaji, tag skill, dan metadata lowongan.
- File output disimpan ke `data/processed/master_dataset_eda.csv` dengan tipe data yang sudah seragam, kolom gaji numerik, dan 32 kolom boolean skill.
- Fungsi utama pembersihan data berada di `src/glints_scraper/transform.py` dan diuji lewat unit test di `tests/test_transform.py`.

---

## Tahapan Proses di Fungsi `transform()`

Fungsi `transform()` memproses data mentah melalui lima tahap berurutan.

1. Menghapus baris data yang tidak lengkap seperti deskripsi kosong atau struktur kolom yang rusak.
2. Menyaring hanya lowongan yang masih aktif dan berlokasi di Indonesia, sekaligus membuang lowongan luar negeri seperti Vietnam atau Singapura serta merapikan nama kota ke kolom `city_clean`.
3. Mengubah teks gaji IDR dari berbagai format menjadi angka bulanan pada kolom `salary_min`, `salary_max`, dan `salary_avg`.
4. Mencocokkan pola regex untuk mendeteksi 32 skill teknis (seperti Python, SQL, Cloud, Tableau, dan Power BI) dari deskripsi dan tag menjadi kolom boolean `skill_<name>`.
5. Menyeragamkan syarat pengalaman kerja ke angka tahun di `exp_min_years`, jenjang pendidikan ke kategori baku (S1, S2, D3, SMA/SMK), tipe kontrak (*Full-time, Contract, Internship*), dan model kerja (*On-site, Hybrid, Remote*).

In [1]:
import sys
from pathlib import Path
import pandas as pd

# Menambahkan direktori src/ ke sys.path agar modul lokal dapat diimpor langsung
sys.path.insert(0, str(Path.cwd() / "src"))

from glints_scraper.transform import transform

print("[INFO] Environment dan modul glints_scraper.transform berhasil dimuat.")

[INFO] Environment dan modul glints_scraper.transform berhasil dimuat.


---

## 1. Memuat Data Mentah

Tahap pertama adalah membaca file CSV hasil scraping untuk mengecek struktur kolom awal sebelum dibersihkan.

In [2]:
raw_path = Path("data/raw/job_details.csv")

if not raw_path.exists():
    print(f"[PERINGATAN] File mentah {raw_path} tidak ditemukan secara lokal.")
    print("Membuat sampel data mentah untuk demonstrasi pipeline...")
    sample_raw = pd.DataFrame({
        "url": ["https://glints.com/id/opportunities/jobs/demo-1"],
        "keyword": ["data analyst"],
        "job_title": ["Data Analyst"],
        "company_name": ["PT Teknologi Nusantara"],
        "job_location": ["Jakarta, Indonesia"],
        "job_description": ["Kandidat wajib menguasai SQL, Python, dan Tableau untuk reporting."],
        "skill_tags": ["SQL, Python, Tableau"],
        "job_salary": ["Rp 8.000.000 - 12.000.000 / bulan"],
        "job_education": ["S1 Sistem Informasi"],
        "job_method": ["Full-time"],
        "job_experience": ["1-2 tahun"]
    })
    raw = sample_raw
else:
    raw = pd.read_csv(raw_path)

print(f"[DATA] Dimensi Raw Data: {raw.shape[0]:,} baris x {raw.shape[1]} kolom")
raw.head()

[DATA] Dimensi Raw Data: 406 baris x 11 kolom


,job_title,company_name,job_location,keyword,url,job_description,skill_tags,job_salary,job_education,job_method,job_experience
0,Data Analyst,PT. Telexindo Bizmart,"Jakarta Selatan , DKI Jakarta",data analyst,https://glints.com/id/opportunities/jobs/data-...,Kualifikasi sebagai Data Analyst :a. Mahir men...,"Data Analysis, SQL, Microsoft Excel, Python",5.300.000 - 5.600.000,Minimal Sarjana (S1),Penuh Waktu · Kerja di lokasi,1 - 3 tahun pengalaman
1,Data Analyst,PT. Meteor Inovasi Digital,"Jakarta Selatan , DKI Jakarta",data analyst,https://glints.com/id/opportunities/jobs/data-...,Responsibilities:\nQualifications:,"Redis, Python, Tableau, Microsoft SQL Server, ...",6.500.000 - 9.000.000,Minimal Sarjana (S1),Kontrak · Kerja di lokasi,1 - 3 tahun pengalaman
2,Data Analyst,PT Pro Sigmaka Mandiri,"Jakarta Selatan , DKI Jakarta",data analyst,https://glints.com/id/opportunities/jobs/data-...,Job Description\nMinimum Qualifications,"Bi Tools, SQL, Microsoft SQL Server",8.000.000 - 10.000.000,Minimal Sarjana (S1),Kontrak · Kerja di lokasi,3 - 5 tahun pengalaman
3,Data Analyst,PT Sigma Global Teknologi,"Jakarta Selatan , DKI Jakarta",data analyst,https://glints.com/id/opportunities/jobs/data-...,Qualifications:,"VBA, SQL, Redis, Microsoft Excel, Data Analysi...",NaN,Minimal Sarjana (S1),Kontrak · Kerja di lokasi,3 - 5 tahun pengalaman
4,Data Analyst,PT Sigma Global Teknologi,"Jakarta Selatan , DKI Jakarta",data analyst,https://glints.com/id/opportunities/jobs/data-...,Can Join Immediately\nRequirements:\n* Minimal...,"Python, Google Analytics, Power BI, Business I...",8.000.000 - 12.000.000,Minimal Sarjana (S1),Kontrak · Kerja di lokasi,3 - 5 tahun pengalaman


---

## 2. Menjalankan Pembersihan Data

Fungsi `transform(raw)` memproses data mentah tanpa mengubah dataframe aslinya dan mengembalikan dataframe baru yang sudah bersih.

In [3]:
master = transform(raw)

print(f"[DATA] Dimensi Master Dataset: {master.shape[0]:,} baris x {master.shape[1]} kolom")
print(f"[KOLOM] Sampel kolom hasil transformasi: {list(master.columns)[:12]}...")
master.head()

[DATA] Dimensi Master Dataset: 400 baris x 45 kolom
[KOLOM] Sampel kolom hasil transformasi: ['keyword', 'job_title', 'company_name', 'city_clean', 'salary_min', 'salary_max', 'salary_avg', 'exp_min_years', 'employment_type', 'workplace_type', 'job_education_clean', 'skill_python']...


,keyword,job_title,company_name,city_clean,salary_min,salary_max,salary_avg,exp_min_years,employment_type,workplace_type,...,skill_kubernetes,skill_git,skill_statistics,skill_machine_learning,skill_deep_learning,skill_nlp,skill_computer_vision,skill_data_visualization,job_location,url
0,data analyst,Data Analyst,PT. Telexindo Bizmart,Jakarta,5300000.0,5600000.0,5450000.0,1,Full-time,On-site,...,False,False,False,False,False,False,False,False,"Jakarta Selatan , DKI Jakarta",https://glints.com/id/opportunities/jobs/data-...
1,data analyst,Data Analyst,PT. Meteor Inovasi Digital,Jakarta,6500000.0,9000000.0,7750000.0,1,Contract,On-site,...,False,False,True,False,False,False,False,True,"Jakarta Selatan , DKI Jakarta",https://glints.com/id/opportunities/jobs/data-...
2,data analyst,Data Analyst,PT Pro Sigmaka Mandiri,Jakarta,8000000.0,10000000.0,9000000.0,3,Contract,On-site,...,False,False,False,False,False,False,False,False,"Jakarta Selatan , DKI Jakarta",https://glints.com/id/opportunities/jobs/data-...
3,data analyst,Data Analyst,PT Sigma Global Teknologi,Jakarta,NaN,NaN,NaN,3,Contract,On-site,...,False,False,True,False,False,False,False,True,"Jakarta Selatan , DKI Jakarta",https://glints.com/id/opportunities/jobs/data-...
4,data analyst,Data Analyst,PT Sigma Global Teknologi,Jakarta,8000000.0,12000000.0,10000000.0,3,Contract,On-site,...,False,False,False,False,False,False,False,True,"Jakarta Selatan , DKI Jakarta",https://glints.com/id/opportunities/jobs/data-...


---

## 3. Pengecekan Skema dan Kualitas Data

Bagian ini memeriksa kelengkapan kolom utama, jumlah fitur skill, dan sebaran kategori untuk memastikan tidak ada tipe data yang salah sebelum dipakai analisis.

In [4]:
expected_core_columns = {
    "keyword", "job_title", "company_name", "city_clean",
    "salary_min", "salary_max", "salary_avg",
    "exp_min_years", "employment_type", "workplace_type",
    "job_education_clean",
}

skill_columns = [col for col in master.columns if col.startswith("skill_")]

print(f"[VALIDASI] Total baris hasil proses: {len(master):,}")
print(f"[VALIDASI] Seluruh kolom inti terpenuhi: {expected_core_columns.issubset(master.columns)}")
print(f"[VALIDASI] Total fitur skill terdeteksi: {len(skill_columns)}")

if "workplace_type" in master.columns and not master["workplace_type"].empty:
    print("\n[DISTRIBUSI] Model Kerja (Workplace Type):")
    print(master["workplace_type"].value_counts().to_string())

if "job_education_clean" in master.columns and not master["job_education_clean"].empty:
    print("\n[DISTRIBUSI] Jenjang Pendidikan Minimal:")
    print(master["job_education_clean"].value_counts().to_string())

[VALIDASI] Total baris hasil proses: 400
[VALIDASI] Seluruh kolom inti terpenuhi: True
[VALIDASI] Total fitur skill terdeteksi: 32

[DISTRIBUSI] Model Kerja (Workplace Type):
workplace_type
On-site    333
Hybrid      47
Remote      20

[DISTRIBUSI] Jenjang Pendidikan Minimal:
job_education_clean
S1            357
D3/Diploma     40
SMA/SMK         3


---

## 4. Menyimpan Dataset Bersih

Data yang sudah lolos pengecekan disimpan ke `data/processed/master_dataset_eda.csv`. File ini langsung dipakai oleh notebook `eda.ipynb`, laporan `eda_report.html`, dan dashboard Streamlit.

In [5]:
output_path = Path("data/processed/master_dataset_eda.csv")

# Proteksi keamanan dataset:
# Jika file master 430 data terverifikasi sudah ada dan notebook dijalankan pada sampel kecil,
# pertahankan dataset 430 data utama agar integritas analisis tidak hilang.
if output_path.exists() and len(master) < 400:
    existing_len = len(pd.read_csv(output_path))
    print(f"[INFO] Master dataset utama sudah tersedia ({existing_len:,} baris).")
    print(f"[INFO] Pipeline demonstrasi sukses berjalan untuk {len(master):,} sampel data mentah.")
    print("[INFO] Dataset produksi tetap dipertahankan aman tanpa risiko penimpaan.")
else:
    output_path.parent.mkdir(parents=True, exist_ok=True)
    master.to_csv(output_path, index=False)
    print(f"[SUKSES] {len(master):,} baris berhasil disimpan ke {output_path}")

[SUKSES] 400 baris berhasil disimpan ke data\processed\master_dataset_eda.csv
